In [1]:
import cv2
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
pivot_x = 947      # pivot that was used when the angles in the csv files were computed
pivot_y = 180

Fixed_pivot_coords = np.array([[916, 316], [924, 315], [922, 314], [933, 306], [935,311], [930,312], [931,311], [924,311],[936,311],[930,311],[926,319],[915,313],[906,313]], dtype=np.float32)

first_vid, last_vid = 8, 20

# column names in your input csv (change if they differ)
col_theta1 = "theta1_rad_filled"
col_theta2 = "theta2_rel_rad_filled"   # now measured relative to the y-axis

start_angles = []

for i, vid_nr in enumerate(range(first_vid, last_vid + 1)):
    filename = f"Meetdag 2/Meetdag2_Vid{vid_nr}_Total.csv"
    coords = pd.read_csv(filename)

    new_pivot_x, new_pivot_y = Fixed_pivot_coords[i]

    # --- arm lengths in pixels, from frames with a valid detection ---
    L1 = np.nanmedian(np.hypot(coords["Gx"] - pivot_x, coords["Gy"] - pivot_y))
    L2 = np.nanmedian(np.hypot(coords["Bx"] - coords["Gx"], coords["By"] - coords["Gy"]))

    # --- 2. positions from the (interpolated) angles, using the original pivot ---
    theta1_old = coords[col_theta1].to_numpy(dtype=float)
    theta2_old = coords[col_theta2].to_numpy(dtype=float)

    Gx = pivot_x + L1 * np.sin(theta1_old)
    Gy = pivot_y + L1 * np.cos(theta1_old)
    Bx = Gx + L2 * np.sin(theta2_old)
    By = Gy + L2 * np.cos(theta2_old)

    # --- 3. new angles with the correct pivot of this video ---
    theta1_new = np.arctan2(Gx - new_pivot_x, Gy - new_pivot_y)
    theta2_new = np.arctan2(Bx - Gx, By - Gy)

    # sanity check: arm 1 length from the new pivot should be nearly constant
    r1_new = np.hypot(Gx - new_pivot_x, Gy - new_pivot_y)
    spread = np.nanstd(r1_new) / np.nanmean(r1_new)

    # --- 4. save ---
    coords_new = pd.DataFrame({
        "frame": coords["frame"],
        "time_s": coords["time_s"],
        "Gx": Gx, "Gy": Gy, "Bx": Bx, "By": By,
        "theta1_rad": theta1_new,
        "theta2_rad": theta2_new,
        "theta1_deg": np.degrees(theta1_new),
        "theta2_deg": np.degrees(theta2_new),
    })
    # keep the interpolation flags if they exist in the input
    for c in coords.columns:
        if c.endswith("_interpolated"):
            coords_new[c] = coords[c]

    out_name = f"Meting_processed_Vid{i + 1}.csv"
    coords_new.to_csv(out_name, index=False)

    # --- 5. starting angle of arm 1 (first valid frame) ---
    first_valid = coords_new["theta1_deg"].first_valid_index()
    start = coords_new.loc[first_valid, "theta1_deg"] if first_valid is not None else np.nan
    start_angles.append((out_name, filename, start))

    print(f"{filename} -> {out_name} | L1={L1:.1f}px, L2={L2:.1f}px, arm 1 length spread={spread:.1%}")

print("\nStarting angle theta1 per file:")
for out_name, src, ang in start_angles:
    print(f"{out_name} (from {src}): {ang:.2f} deg")

Meetdag 2/Meetdag2_Vid8_Total.csv -> Meting_processed_Vid1.csv | L1=492.0px, L2=239.0px, arm 1 length spread=13.4%
Meetdag 2/Meetdag2_Vid9_Total.csv -> Meting_processed_Vid2.csv | L1=484.4px, L2=238.5px, arm 1 length spread=14.4%
Meetdag 2/Meetdag2_Vid10_Total.csv -> Meting_processed_Vid3.csv | L1=487.4px, L2=238.8px, arm 1 length spread=11.2%
Meetdag 2/Meetdag2_Vid11_Total.csv -> Meting_processed_Vid4.csv | L1=486.0px, L2=239.0px, arm 1 length spread=7.8%
Meetdag 2/Meetdag2_Vid12_Total.csv -> Meting_processed_Vid5.csv | L1=480.2px, L2=239.7px, arm 1 length spread=11.5%
Meetdag 2/Meetdag2_Vid13_Total.csv -> Meting_processed_Vid6.csv | L1=473.1px, L2=238.4px, arm 1 length spread=9.1%
Meetdag 2/Meetdag2_Vid14_Total.csv -> Meting_processed_Vid7.csv | L1=477.2px, L2=238.6px, arm 1 length spread=8.5%
Meetdag 2/Meetdag2_Vid15_Total.csv -> Meting_processed_Vid8.csv | L1=469.9px, L2=238.3px, arm 1 length spread=9.0%
Meetdag 2/Meetdag2_Vid16_Total.csv -> Meting_processed_Vid9.csv | L1=477.2px, 